In [0]:
%sql
CREATE CATALOG IF NOT EXISTS de_etl_project;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS de_etl_project.hdb;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS de_etl_project.hdb.raw;

In [0]:
# load raw files into dataframes
df = spark.read.csv(f"/Volumes/de_etl_project/hdb/raw/1990-1999.csv", header=True, inferSchema=True, sep=",")
df2 = spark.read.csv(f"/Volumes/de_etl_project/hdb/raw/2000-2012.csv", header=True, inferSchema=True, sep=",")
df3 = spark.read.csv(f"/Volumes/de_etl_project/hdb/raw/2012-2014.csv", header=True, inferSchema=True, sep=",")
df4 = spark.read.csv(f"/Volumes/de_etl_project/hdb/raw/2015-2016.csv", header=True, inferSchema=True, sep=",")
df5 = spark.read.csv(f"/Volumes/de_etl_project/hdb/raw/2017.csv", header=True, inferSchema=True, sep=",")

In [0]:
# check for number of columns in each file
print(f'''1990-1999.csv: {len(df.columns)}
2000-2012.csv: {len(df2.columns)}
2012-2014.csv: {len(df3.columns)}
2015-2016.csv: {len(df4.columns)}
2017.csv: {len(df5.columns)}
''')

1990-1999.csv: 10
2000-2012.csv: 10
2012-2014.csv: 10
2015-2016.csv: 11
2017.csv: 11



In [0]:
# check for column headers
print(f'''1990-1999.csv: {df.columns}
2000-2012.csv: {df2.columns}
2012-2014.csv: {df3.columns}
2015-2016.csv: {df4.columns}
2017-2026.csv: {df5.columns}
''')

1990-1999.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
2000-2012.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
2012-2014.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
2015-2016.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']
2017-2026.csv: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']



In [0]:
df.printSchema

<bound method DataFrame.printSchema of DataFrame[month: date, town: string, flat_type: string, block: string, street_name: string, storey_range: string, floor_area_sqm: double, flat_model: string, lease_commence_date: int, resale_price: int]>

In [0]:
# df before addition of new column
display(df.head(10))
display(df2.head(10))
display(df3.head(10))

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
2012-03-01,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45.0,Improved,1986,250000.0
2012-03-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44.0,Improved,1980,265000.0
2012-03-01,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68.0,New Generation,1980,315000.0
2012-03-01,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1984,320000.0
2012-03-01,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67.0,New Generation,1980,321000.0
2012-03-01,ANG MO KIO,3 ROOM,154,ANG MO KIO AVE 5,01 TO 05,68.0,New Generation,1981,321000.0
2012-03-01,ANG MO KIO,3 ROOM,110,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1978,323000.0
2012-03-01,ANG MO KIO,3 ROOM,445,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1979,325000.0
2012-03-01,ANG MO KIO,3 ROOM,476,ANG MO KIO AVE 10,11 TO 15,67.0,New Generation,1979,328000.0
2012-03-01,ANG MO KIO,3 ROOM,631,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1985,330000.0


month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
2000-01-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0
2000-01-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0
2000-01-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0
2000-01-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0
2000-01-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0
2000-01-01,ANG MO KIO,3 ROOM,320,ANG MO KIO AVE 1,04 TO 06,73.0,New Generation,1977,157000.0
2000-01-01,ANG MO KIO,3 ROOM,320,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1977,178000.0
2000-01-01,ANG MO KIO,3 ROOM,330,ANG MO KIO AVE 1,07 TO 09,68.0,New Generation,1981,160000.0
2000-01-01,ANG MO KIO,3 ROOM,330,ANG MO KIO AVE 1,04 TO 06,68.0,New Generation,1981,169000.0
2000-01-01,ANG MO KIO,3 ROOM,332,ANG MO KIO AVE 1,07 TO 09,82.0,New Generation,1981,205000.0


month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
2012-03-01,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45.0,Improved,1986,250000.0
2012-03-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44.0,Improved,1980,265000.0
2012-03-01,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68.0,New Generation,1980,315000.0
2012-03-01,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1984,320000.0
2012-03-01,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67.0,New Generation,1980,321000.0
2012-03-01,ANG MO KIO,3 ROOM,154,ANG MO KIO AVE 5,01 TO 05,68.0,New Generation,1981,321000.0
2012-03-01,ANG MO KIO,3 ROOM,110,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1978,323000.0
2012-03-01,ANG MO KIO,3 ROOM,445,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1979,325000.0
2012-03-01,ANG MO KIO,3 ROOM,476,ANG MO KIO AVE 10,11 TO 15,67.0,New Generation,1979,328000.0
2012-03-01,ANG MO KIO,3 ROOM,631,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1985,330000.0


In [0]:
from pyspark.sql import functions as f

dfs = [df,df2,df3]
updated_dfs = []

for i, df in enumerate(dfs):
    # print(dfs[i])
    # remaning_lease can be calculated using 99 - (year - lease_commence_date) for df, df2, df3
    dfs[i] = df.withColumn(
        "remaining_lease",
        99 - ( f.year('month') - f.col('lease_commence_date') )
    )

df, df2, df3 = dfs

print(df.columns)
print(df2.columns)
print(df3.columns)

['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']


In [0]:
# df after addition of new column
display(df.head(10))
display(df2.head(10))
display(df3.head(10))

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
2012-03-01,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45.0,Improved,1986,250000.0,73
2012-03-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44.0,Improved,1980,265000.0,67
2012-03-01,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68.0,New Generation,1980,315000.0,67
2012-03-01,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1984,320000.0,71
2012-03-01,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67.0,New Generation,1980,321000.0,67
2012-03-01,ANG MO KIO,3 ROOM,154,ANG MO KIO AVE 5,01 TO 05,68.0,New Generation,1981,321000.0,68
2012-03-01,ANG MO KIO,3 ROOM,110,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1978,323000.0,65
2012-03-01,ANG MO KIO,3 ROOM,445,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1979,325000.0,66
2012-03-01,ANG MO KIO,3 ROOM,476,ANG MO KIO AVE 10,11 TO 15,67.0,New Generation,1979,328000.0,66
2012-03-01,ANG MO KIO,3 ROOM,631,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1985,330000.0,72


month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
2000-01-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,85
2000-01-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,85
2000-01-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,75
2000-01-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,75
2000-01-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,75
2000-01-01,ANG MO KIO,3 ROOM,320,ANG MO KIO AVE 1,04 TO 06,73.0,New Generation,1977,157000.0,76
2000-01-01,ANG MO KIO,3 ROOM,320,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1977,178000.0,76
2000-01-01,ANG MO KIO,3 ROOM,330,ANG MO KIO AVE 1,07 TO 09,68.0,New Generation,1981,160000.0,80
2000-01-01,ANG MO KIO,3 ROOM,330,ANG MO KIO AVE 1,04 TO 06,68.0,New Generation,1981,169000.0,80
2000-01-01,ANG MO KIO,3 ROOM,332,ANG MO KIO AVE 1,07 TO 09,82.0,New Generation,1981,205000.0,80


month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
2012-03-01,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45.0,Improved,1986,250000.0,73
2012-03-01,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44.0,Improved,1980,265000.0,67
2012-03-01,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68.0,New Generation,1980,315000.0,67
2012-03-01,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1984,320000.0,71
2012-03-01,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67.0,New Generation,1980,321000.0,67
2012-03-01,ANG MO KIO,3 ROOM,154,ANG MO KIO AVE 5,01 TO 05,68.0,New Generation,1981,321000.0,68
2012-03-01,ANG MO KIO,3 ROOM,110,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1978,323000.0,65
2012-03-01,ANG MO KIO,3 ROOM,445,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1979,325000.0,66
2012-03-01,ANG MO KIO,3 ROOM,476,ANG MO KIO AVE 10,11 TO 15,67.0,New Generation,1979,328000.0,66
2012-03-01,ANG MO KIO,3 ROOM,631,ANG MO KIO AVE 4,01 TO 05,67.0,New Generation,1985,330000.0,72


In [0]:
# check for number of columns in each file
print(f'''1990-1999.csv: {len(df.columns)}
2000-2012.csv: {len(df2.columns)}
2012-2014.csv: {len(df3.columns)}
2015-2016.csv: {len(df4.columns)}
2017.csv: {len(df5.columns)}
''')

1990-1999.csv: 11
2000-2012.csv: 11
2012-2014.csv: 11
2015-2016.csv: 11
2017.csv: 11

